In [0]:
# metric table
silver_nss_themes = (
    spark.read
    .format("delta")
    .load("abfss://silver-tables@jdnhsbronze.dfs.core.windows.net/themes_nss/")
)
people_promise = (
    spark.read
    .format("delta")
    .load("abfss://silver-tables@jdnhsbronze.dfs.core.windows.net/people_promise_nss/")
)
silver_nss_themes.show(10)
people_promise.show(10)

+--------+--------------------+--------------------+----------+--------------------+----------------+----------------+--------------------+-----+---------------+--------------------+------------+--------------------+
|ods_code|            org_name|  benchmarking_group|    region|                 ics|weighting_method|     metric_code|        metric_label|score|total_responses|         source_file|source_sheet|      load_timestamp|
+--------+--------------------+--------------------+----------+--------------------+----------------+----------------+--------------------+-----+---------------+--------------------+------------+--------------------+
|   G6V2S|north london nhs ...|mental health & l...|    london|north central london|occupation group|STAFF ENGAGEMENT|    staff engagement| 6.59|           2268|nss25_detailed_sp...|      themes|2026-09-13 14:17:...|
|   G6V2S|north london nhs ...|mental health & l...|    london|north central london|occupation group|             E_1|          moti

In [0]:

from pyspark.sql.functions import col, lower,trim, row_number

silver_nss_themes = (silver_nss_themes
    .select(col("metric_label"), col("metric_code"))
    .distinct() )

silver_people_promise = (people_promise  
    .select(col("metric_label"), col("metric_code"))
    .distinct())


    
metric_gold = (silver_nss_themes.unionByName(silver_people_promise))

metric_gold.show()

+--------------------+----------------+
|        metric_label|     metric_code|
+--------------------+----------------+
|          motivation|             E_1|
|         involvement|             E_2|
|              morale|          MORALE|
|       work pressure|             M_2|
|           stressors|             M_3|
|            advocacy|             E_3|
|thinking about le...|             M_1|
|    staff engagement|STAFF ENGAGEMENT|
|compassionate cul...|           PP1_1|
|             burnout|           PP4_2|
|we each have a vo...|             PP3|
|autonomy and control|           PP3_1|
|negative experiences|           PP4_3|
|          appraisals|           PP5_2|
|        team working|           PP7_1|
|we are compassion...|             PP1|
|diversity and equ...|           PP1_3|
|         development|           PP5_1|
|     line management|           PP7_2|
|           inclusion|           PP1_4|
+--------------------+----------------+
only showing top 20 rows


In [0]:
(metric_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "path",
        "abfss://gold@jdnhsbronze.dfs.core.windows.net/dim_metric/"
    ) \
    .saveAsTable(
        "metric_dimension"
    ))